# GATE PDF Ingestion via MinerU (GPU Accelerated)

This notebook clones your GitHub repository (just to get the PDFs), extracts the PDFs using a GPU, and automatically commits the resulting JSON and Markdown files back to a new branch in your GitHub repository.

**Before running:**
1. Go to `Runtime` -> `Change runtime type` and ensure **T4 GPU** is selected.
2. Add your GitHub Personal Access Token below.

In [ ]:
# 1. Setup GitHub Credentials & Clone
GITHUB_USERNAME = "MarganDas" # Change this to your username
GITHUB_TOKEN = "" # Paste your GitHub Personal Access Token here
REPO_NAME = "GATE"

import os
if not os.path.exists(REPO_NAME):
    !git clone https://{GITHUB_TOKEN}@github.com/{GITHUB_USERNAME}/{REPO_NAME}.git
%cd {REPO_NAME}


In [ ]:
# 2. Install MinerU & Dependencies
!pip install -U magic-pdf[full] transformers==4.49.0 huggingface_hub PyMuPDF


In [ ]:
# 3. Download Models and Setup Config
import os
import json
from huggingface_hub import snapshot_download

models_dir = os.path.abspath("models")
os.makedirs(models_dir, exist_ok=True)
snapshot_download('opendatalab/pdf-extract-kit-1.0', local_dir=models_dir)

# The HuggingFace repo contains a folder called 'models' inside it
config = {
  "models-dir": os.path.join(models_dir, "models"),
  "device-mode": "cuda",
  "layout-config": { "model": "doclayout_yolo" }
}
with open("magic-pdf.json", "w") as f:
    json.dump(config, f, indent=4)


In [ ]:
# 4. Split and Run MinerU Extraction (RAM Optimization)
import glob
import os
import shutil
import fitz # PyMuPDF

os.environ["MINERU_TOOLS_CONFIG_JSON"] = os.path.abspath("magic-pdf.json")

pdf_files = glob.glob("filter1_*.pdf")
if not pdf_files:
    raise Exception("❌ NO PDF FILES FOUND! Make sure the PDFs are pushed to GitHub or uploaded directly to this folder.")

if os.path.exists("data/raw_extraction"):
    shutil.rmtree("data/raw_extraction")

# Fix for PaddleOCR version mismatch
!ln -sf /content/GATE/models/models/OCR/paddleocr_torch/Multilingual_PP-OCRv3_det_infer.pth /content/GATE/models/models/OCR/paddleocr_torch/ch_PP-OCRv3_det_infer.pth

CHUNK_SIZE = 50  # Process 50 pages at a time to prevent Colab RAM crashes
pdf_chunks_map = {}

for pdf in pdf_files:
    print(f"\n{'='*50}\nSplitting and Processing {pdf}...\n{'='*50}")
    doc = fitz.open(pdf)
    num_pages = len(doc)
    base_name = pdf.replace(".pdf", "")
    chunk_paths = []
    
    for i in range(0, num_pages, CHUNK_SIZE):
        start_page = i
        end_page = min(i + CHUNK_SIZE - 1, num_pages - 1)
        chunk_pdf = f"{base_name}_part{i//CHUNK_SIZE}.pdf"
        
        # Create a new PDF for this chunk
        chunk_doc = fitz.open()
        chunk_doc.insert_pdf(doc, from_page=start_page, to_page=end_page)
        chunk_doc.save(chunk_pdf)
        chunk_doc.close()
        chunk_paths.append(chunk_pdf)
        
        print(f"\n---> Processing Chunk {i//CHUNK_SIZE + 1} (Pages {start_page+1} to {end_page+1}) <--- \n")
        !magic-pdf -p "{chunk_pdf}" -o "data/raw_extraction"
        
    doc.close()
    pdf_chunks_map[base_name] = chunk_paths


In [ ]:
# 5. Custom Parser Logic (Combines Chunks)
import re
import json
import os

def enrich_markdown_with_bboxes(md_path, middle_json_path):
    with open(md_path, 'r', encoding='utf-8') as f:
        md_text = f.read()
    with open(middle_json_path, 'r', encoding='utf-8') as f:
        middle = json.load(f)
    image_map = {}
    for page_idx, page_info in enumerate(middle.get("pdf_info", [])):
        def find_images(obj, current_bbox=None):
            if isinstance(obj, dict):
                bbox = obj.get("bbox", current_bbox)
                if "image_path" in obj:
                    image_map[obj["image_path"]] = {"page": page_idx, "bbox": bbox}
                for v in obj.values():
                    find_images(v, bbox)
            elif isinstance(obj, list):
                for item in obj:
                    find_images(item, current_bbox)
        for block in page_info.get("preproc_blocks", []):
            find_images(block, block.get("bbox"))
    for img_name, data in image_map.items():
        md_img_str1 = f"![](images/{img_name})"
        md_img_str2 = f"![]({img_name})"
        pointer = json.dumps({"pdf_loc": {"page": data["page"], "bbox": data["bbox"]}})
        replacement = f"[IMAGE_POINTER: {pointer}]"
        md_text = md_text.replace(md_img_str1, replacement)
        md_text = md_text.replace(md_img_str2, replacement)
    return md_text

def parse_markdown_to_json(md_text):
    structured_data = []
    sections = re.split(r'(?m)^#{1,3}\s+(.+)$', md_text)
    current_topic = "General"
    if sections[0].strip():
        structured_data.append({"type": "note", "topic": current_topic, "content": sections[0].strip()})
    for i in range(1, len(sections), 2):
        if i + 1 >= len(sections): break
        header, content = sections[i].strip(), sections[i+1].strip()
        if not content:
            current_topic = header
            continue
        if "table of contents" in header.lower() or "contributors" in header.lower(): continue
        is_question, options_text, answer_text = False, None, None
        if re.search(r'\bA\.\s*.*\bB\.\s*', content) or 'gate' in header.lower():
            is_question = True
        ans_match = re.search(r'#+\s*Answer\s*key.*?([A-D])', content, re.IGNORECASE | re.DOTALL)
        if ans_match:
            answer_text = ans_match.group(1)
            is_question = True
        if "question" in header.lower(): is_question = True
        if is_question:
            structured_data.append({"type": "question", "topic": header if "question" not in header.lower() else current_topic, "question_text": content, "options": None, "answer": answer_text, "solution": None})
        else:
            structured_data.append({"type": "note", "topic": header, "content": content})
        current_topic = header
    return structured_data

parsed_any = False
os.makedirs("data", exist_ok=True)

for base_name, chunk_paths in pdf_chunks_map.items():
    combined_structured_data = []
    for chunk_pdf in chunk_paths:
        chunk_name = chunk_pdf.replace(".pdf", "")
        pdf_out_dir = os.path.join("data/raw_extraction", chunk_name, "auto")
        md_file = os.path.join(pdf_out_dir, f"{chunk_name}.md")
        middle_json_file = os.path.join(pdf_out_dir, f"{chunk_name}_middle.json")
        
        if os.path.exists(md_file):
            parsed_any = True
            print(f"Parsing {md_file}...")
            enriched = enrich_markdown_with_bboxes(md_file, middle_json_file)
            structured = parse_markdown_to_json(enriched)
            combined_structured_data.extend(structured)
        else:
            print(f"❌ ERROR: Missing markdown file {md_file}. Magic-PDF failed silently on this chunk.")
            
    out_json = os.path.join("data", f"{base_name}_structured.json")
    print(f"Saving combined JSON to {out_json}...")
    with open(out_json, "w", encoding="utf-8") as f:
        json.dump(combined_structured_data, f, indent=4, ensure_ascii=False)

if not parsed_any:
    raise Exception("Nothing was parsed!")


In [ ]:
# 6. Commit and Push back to GitHub
!git config --global user.email "colab@example.com"
!git config --global user.name "Colab Bot"
!git checkout -b extracted-data-update
!git add -f data/  # Use -f because data/ might be in your .gitignore
!git commit -m "Automated PDF extraction via Google Colab"
!git push origin extracted-data-update --force
print("✅ Successfully pushed to the 'extracted-data-update' branch!")
